In [1]:
import os
from google import genai

gemini_client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

## PREPARATION

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

## QUESTION 1

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)
    
print(f"Total Pages: {len(documents)}")

Total Pages: 72


## QUESTION 2

In [4]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

question = "How does the agentic loop keep calling the model until it stops?"
search_result = index.search(question, num_results=5)

# first result
print(f"filename in first result: {search_result[0]['filename']}")

filename in first result: 01-agentic-rag/lessons/14-agentic-loop.md


### Call Template Function rag_helper.py

In [12]:
!curl -L https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py -o rag_helper.py

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
100   2134 100   2134   0      0   3844      0                              0
100   2134 100   2134   0      0   3842      0                              0
100   2134 100   2134   0      0   3841      0                              0


## QUESTION 3

In [5]:
from rag_helper import RAGBase

RAG = RAGBase(index, gemini_client)

query = "How does the agentic loop keep calling the model until it stops?"
response = RAG.rag(query)

print(response.text)

print("Input tokens :", response.usage_metadata.prompt_token_count)
print("Output tokens:", response.usage_metadata.candidates_token_count)
print("Total tokens :", response.usage_metadata.total_token_count)


The agentic loop keeps calling the model until it stops by wrapping the process in a `while` loop. This loop continues to:

1.  **Call the model:** It sends the current message history to the LLM.
2.  **Process the response:**
    *   If the model's response contains a `function_call`, the agent executes that function (e.g., `search`).
    *   The result of the function call is then appended to the message history.
    *   A flag (`has_function_calls`) is set to `True`.
    *   If the model returns a message (an answer), it is printed.
3.  **Loop or stop:** The loop continues as long as `has_function_calls` is `True`, meaning the model requested a tool call and its output hasn't been seen by the model yet. The loop *breaks* when the model returns a response that **does not contain any function calls**, indicating that it has finished its task and provided a final answer.

Essentially, the model is in control; it decides when to call tools and how many times to do so, and the loop conti

## QUESTION 4

In [6]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Total chunks generated: {len(chunks)}")

Total chunks generated: 295


## QUESTION 5

In [7]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)

RAG = RAGBase(index, gemini_client)

query = "How does the agentic loop keep calling the model until it stops?"
response_chunk = RAG.rag(query)

print("Input tokens (Question 3) :", response.usage_metadata.prompt_token_count)
print("Input tokens (Question 5) :", response_chunk.usage_metadata.prompt_token_count)


Input tokens (Question 3) : 7958
Input tokens (Question 5) : 2611


Because Token Q5 divide by Token Q3; we get result where token Q5 3 x fewer than Q3 

Answer : 3 x fewer

## QUESTION 6

In [8]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [9]:
## Last index is from chunks method

def search(query):
    boost_dict = {'content':3.0, 'filename':2.0}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [10]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [11]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [ ]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    #Count agent call search
    search.count += 1
    
    return index.search(
        query,
        num_results=5,
        boost_dict={'content':3.0, 'filename':2.0}
    )
search.count = 0

In [13]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [14]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [15]:
from google.genai import types

AGENT_PROMPT = """
You're a course teaching assistant. 
Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
"""

def ask_agent(question):
    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=question,
        config=types.GenerateContentConfig(
            system_instruction=AGENT_PROMPT,
            tools=[search]
        )
    )
    return response.text

In [16]:
answer = ask_agent("How does the agentic loop work, and how is it different from plain RAG?")
print(f"Total Agent Search Calls: {search.count}")
print(answer)

Total Agent Search Calls: 2
The agentic loop and plain Retrieval Augmented Generation (RAG) are both techniques to enhance Large Language Model (LLM) capabilities, but they differ significantly in their approach to control flow, decision-making, and iteration.

### Plain RAG

Plain RAG follows a fixed, three-step pipeline:
1.  **Search/Retrieve:** Given a user's question, relevant information is retrieved from a data source (e.g., a vector database) using a search mechanism (like keyword or vector search).
2.  **Augment:** The retrieved information is then added to the original user query to create an augmented prompt.
3.  **Generate:** The LLM receives this augmented prompt and generates a response based on the provided context.

The primary goal of plain RAG is to ground the LLM's answers in specific, up-to-date data, thereby reducing hallucinations and improving factual accuracy. It's a sequential process where the retrieval step happens *before* the LLM generates its final output.
